# 01 — Carregamento e Limpeza dos Dados

**Dataset:** Microdados do ENEM 2023 (INEP)  
**Objetivo:** Carregar uma amostra de 500 mil candidatos, inspecionar qualidade dos dados, aplicar limpeza e exportar o dataset processado para os notebooks seguintes.

**Etapas:**
1. Carregamento direto do arquivo ZIP
2. Inspeção inicial (shape, tipos, primeiros registros)
3. Análise de valores ausentes
4. Filtragem e transformação
5. Exportação para `data/processed/`

In [ ]:
import sys
import zipfile
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '../src')
from utils import (
    NOTAS_COLS, NOTAS_LABELS,
    MAPA_ESCOLA, MAPA_COR_RACA, MAPA_RENDA,
    configurar_estilo, salvar_figura
)

configurar_estilo()

## 1. Carregamento dos Dados

In [ ]:
ZIP_PATH       = '../data/raw/microdados_enem_2023.zip'
PROCESSED_PATH = '../data/processed/amostra_limpa.parquet'
N_AMOSTRA      = 500_000

COLUNAS_INTERESSE = [
    'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO',
    'Q006',        # faixa de renda familiar
    'TP_ESCOLA',   # tipo de escola
    'SG_UF_PROVA', # estado
    'TP_SEXO',     # sexo
    'TP_COR_RACA', # cor/raça autodeclarada
]

In [ ]:
print('Carregando dados do arquivo ZIP...')
t0 = time.time()

with zipfile.ZipFile(ZIP_PATH) as z:
    # O ZIP contém também ITENS_PROVA; seleciona o maior CSV (MICRODADOS_ENEM_2023.csv)
    csvs = [f for f in z.namelist() if f.lower().endswith('.csv')]
    csv_nome = max(csvs, key=lambda f: z.getinfo(f).file_size)
    print(f'  Arquivo CSV encontrado: {csv_nome}')

    with z.open(csv_nome) as f:
        df_raw = pd.read_csv(
            f,
            sep=';',
            encoding='latin-1',
            usecols=COLUNAS_INTERESSE,
            nrows=N_AMOSTRA,
            low_memory=False,
        )

print(f'  Tempo de leitura: {time.time() - t0:.1f}s')
print(f'  Shape carregado: {df_raw.shape}')

## 2. Inspeção Inicial

In [ ]:
df_raw.head(3)

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe().round(2)

## 3. Análise de Valores Ausentes

No ENEM, um candidato inscrito pode não comparecer às provas — nesse caso todas as notas ficam como `NaN`. Candidatos ausentes não são úteis para análise de desempenho e precisam ser removidos.

In [ ]:
ausentes = df_raw.isna().sum().sort_values(ascending=False)
ausentes_pct = (ausentes / len(df_raw) * 100).round(2)

resumo_ausentes = pd.DataFrame({
    'Ausentes': ausentes,
    '% Ausentes': ausentes_pct
})
print(resumo_ausentes.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

cores = ['#d62728' if p > 20 else '#1f77b4' for p in ausentes_pct.values]
bars = ax.barh(ausentes_pct.index, ausentes_pct.values, color=cores)

for bar, val in zip(bars, ausentes_pct.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=9)

ax.set_xlabel('% de valores ausentes')
ax.set_title('Valores Ausentes por Coluna — Amostra Bruta (500k registros)', fontsize=12)
ax.axvline(20, color='red', linestyle='--', linewidth=0.8, alpha=0.6, label='Limite 20%')
ax.legend()
plt.tight_layout()
salvar_figura(fig, '01_valores_ausentes')
plt.show()

## 4. Filtragem e Transformação

### 4.1 — Remover ausentes nas provas

**`NaN` ≠ `0` no dataset do ENEM:**

| Valor | Significado |
|---|---|
| `NaN` | Candidato **ausente** — não compareceu naquele dia de prova |
| `0.0` | Candidato **presente** — compareceu mas zerou a nota |

O filtro abaixo remove apenas os `NaN` (ausentes). Candidatos com nota `0` são mantidos, pois participaram do exame.

**Estrutura dos dias de prova:**
- **Dia 1** → CN + MT (2 notas)
- **Dia 2** → CH + LC + Redação (3 notas)

Como CN e MT são sempre aplicadas juntas, e CH/LC/Redação também, o critério `>= 4` notas preenchidas na prática **mantém apenas candidatos que compareceram aos dois dias**. Candidatos que foram só a um dia têm no máximo 3 notas preenchidas e são excluídos.

Essa decisão garante que a comparação de desempenho seja feita em base igual — apenas quem realizou o exame completo.

In [ ]:
df = df_raw[df_raw[NOTAS_COLS].notna().sum(axis=1) >= 4].copy()

removidos = len(df_raw) - len(df)
print(f'Registros removidos (ausentes): {removidos:,} ({removidos / len(df_raw) * 100:.1f}%)')
print(f'Registros restantes:            {len(df):,}')

### 4.2 — Aplicar mapeamentos de categorias

In [ ]:
df['TP_ESCOLA']   = df['TP_ESCOLA'].map(MAPA_ESCOLA)
df['TP_COR_RACA'] = df['TP_COR_RACA'].map(MAPA_COR_RACA)
df['Q006']        = df['Q006'].map(MAPA_RENDA)
df['TP_SEXO']     = df['TP_SEXO'].map({'M': 'Masculino', 'F': 'Feminino'})

print('Tipos após mapeamento:')
print(df.dtypes)

### 4.3 — Criar coluna de nota média

In [ ]:
df['NOTA_MEDIA'] = df[NOTAS_COLS].mean(axis=1).round(2)

print('Estatísticas da nota média:')
print(df['NOTA_MEDIA'].describe().round(2))

### 4.4 — Converter colunas categóricas

In [ ]:
cols_cat = ['TP_ESCOLA', 'TP_COR_RACA', 'Q006', 'TP_SEXO', 'SG_UF_PROVA']
for col in cols_cat:
    df[col] = df[col].astype('category')

print('Shape final:', df.shape)
print(f'Uso de memória: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

## 5. Verificação Final

In [ ]:
print('=== Dataset limpo ===')
print(f'Shape: {df.shape}')
print()
print('Distribuição por tipo de escola:')
print(df['TP_ESCOLA'].value_counts())
print()
print('Distribuição por sexo:')
print(df['TP_SEXO'].value_counts())
print()
print('Ausentes remanescentes:')
print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
df.head(3)

## 6. Exportação

In [ ]:
df.to_parquet(PROCESSED_PATH, index=False, engine='pyarrow')
print(f'Dataset salvo em: {PROCESSED_PATH}')
print(f'Shape exportado:  {df.shape}')

---
## Resumo

| Etapa | Resultado |
|---|---|
| Registros carregados | 500.000 |
| Removidos (ausentes nas provas) | ver célula acima |
| Colunas no dataset final | 11 (10 originais + `NOTA_MEDIA`) |
| Formato de saída | Parquet (menor, mais rápido que CSV) |

**Próximo passo:** `02_analise_univariada.ipynb` — distribuições das notas e variáveis categóricas.